# Saving the Tokens: Efficient Agent Design
IEEE AgenticAI Summit 2026, Bangalore. Shyamji Tiwari.

**Nothing to install.** This notebook runs in your browser on Google Colab.
Click the play button on each cell, top to bottom. No API key, no model, no cost.

> **Your bill = how big each item is × how many times you send it again.**
> Find the item that dominates your bill, then fix that one.

## 1. Get the files (run once, takes a few seconds)

In [ ]:
#@title Run this first
import os, subprocess, sys

base = "/content" if os.path.isdir("/content") else os.getcwd()   # Colab, or a local Jupyter
repo = os.path.join(base, "agentic-ai-26")
if not os.path.isdir(repo):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ieee-ai/agentic-ai-26.git", repo], check=True)
work = os.path.join(repo, "Saving-the-Tokens")
os.chdir(work)
sys.path.insert(0, work)
print("Ready. Files are in", work)

In [ ]:
#@title Check the setup: look for PASS
!python3 check_setup.py

## 2. Exercise 1: what does one screenshot cost?
Type your own screen size (Mac: menu > About This Mac. Windows: Settings > Display).
Run it, then **double the width**, then try **3840 x 2160**. What moved, and what did not?

In [ ]:
#@title Price one screenshot
width = 1920 #@param {type:"integer"}
height = 1080 #@param {type:"integer"}
!python3 ex1_image.py --size {width}x{height}

## 3. Exercise 2: what does speech cost?

In [ ]:
#@title Price some speech
seconds = 31 #@param {type:"number"}
!python3 ex2_audio.py --seconds {seconds}

## 4. Exercise 3: which fix saves the most?
**Write your prediction first.** The cell after this one prints the answer.
Ten fixes are on the slide: which saves the most, and which saves the least?

In [ ]:
#@title Your prediction (type before you run the next cell)
saves_the_most = "" #@param {type:"string"}
saves_the_least = "" #@param {type:"string"}
print("Locked in:", saves_the_most or "(blank)", "/", saves_the_least or "(blank)")

In [ ]:
#@title Reveal: the screen agent
!python3 ex3_budget.py

### The twist
Same ten fixes, a different agent: a text-only chat agent with a 6,000-token system prompt.
Before you run it: will "cut the system prompt" go **up** or **down** the ranking?

In [ ]:
#@title Two agents side by side
!python3 ex3_budget.py --agent both

## 5. Play: change the assumptions
Change a number, run the cell, and watch the ranking move.
Try a **4,000-token** prompt, then turn **caching** on, then price **output at 5×**.

In [ ]:
#@title Your own agent
agent = "screen" #@param ["screen", "chat"]
system_prompt_tokens = 400 #@param {type:"integer"}
turns = 20 #@param {type:"integer"}
prefix_cache_discount = 0.0 #@param {type:"slider", min:0, max:0.95, step:0.05}
output_price_multiple = 0 #@param {type:"slider", min:0, max:10, step:1}

import ex3_budget as e
base = e.AGENTS[agent][0]
cfg = e.variant(base, {"system_prompt_tokens": system_prompt_tokens, "turns": turns,
                       "prefix_cache_discount": prefix_cache_discount,
                       "output_price_multiple": output_price_multiple}, "mine")
total, scored = e.rank_levers(cfg)
print(f"Session bill: {total:,} tokens   ({agent} agent)\n")
for saving, label, new_total, _final, _costs in scored:
    already = label.startswith("Turn on prefix caching") and prefix_cache_discount > 0
    note = "" if abs(saving) > 0.0005 else ("   (already on)" if already else "   (does not apply to this agent)")
    print(f"{saving * 100:7.1f}%  {label}{note}")

## 6. Take it home: read your own agent's token log
Every model response carries token counts (the `usage` field). Log them per call into a CSV with
`turn,input_tokens,output_tokens` and run the report. Try the samples first.

In [ ]:
#@title Report on a sample log
sample = "sample_usage.csv" #@param ["sample_usage.csv", "sample_usage_chat.csv", "sample_usage_pruned.csv"]
!python3 bill_report.py {sample}

In [ ]:
#@title Or upload your own CSV
try:
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        !python3 bill_report.py "{name}"
except ImportError:
    print("Upload works on Colab. Locally, run: python3 bill_report.py your_log.csv")

---
Files, slides and this notebook: [github.com/ieee-ai/agentic-ai-26/Saving-the-Tokens](https://github.com/ieee-ai/agentic-ai-26/tree/main/Saving-the-Tokens)
The simulation uses illustrative settings, not a measurement of any shipped model.